In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/KAIST/RA'

Mounted at /content/drive
/content/drive/MyDrive/KAIST/RA


In [ ]:
#!pip install spacy beautifulsoup4 requests
#!pip install lxml beautifulsoup4 requests
import requests
from bs4 import BeautifulSoup
import pandas as pd
import spacy
import re
from lxml import etree

In [ ]:
df = pd.read_csv('FINALdata.csv')
df

,Key,Item Type,Publication Year,Author 1,Author 2,Author 3,Author 4,Author 5,Author 6,Author 7,...,Volume,Publisher,Place,Library Catalog,Google Citation,Notes,File Attachments,Book Author,extracted_text,Keywords
0,54ZC89CL,journalArticle,2018.0,"Ferràs-Hernández, Xavier",NaN,NaN,NaN,NaN,NaN,NaN,...,27.0,NaN,NaN,DOI.org (Crossref),90,NaN,C:\\Users\\user\\Zotero\\storage\\LG75XGQE\\Fe...,NaN,NaN,"artificial intelligence,exponential technologi..."
1,AX85B2EL,journalArticle,2021.0,"Fayard, Anne-Laure",NaN,NaN,NaN,NaN,NaN,NaN,...,30.0,NaN,NaN,DOI.org (Crossref),54,NaN,C:\\Users\\user\\Zotero\\storage\\6L72ALKF\\Fa...,NaN,NaN,"philosophy of science,qualitative research,qua..."
2,U9TGTFQL,journalArticle,2023.0,"Faulconbridge, James","Sarwar, Atif","Spring, Martin",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,DOI.org (Crossref),NaN,NaN,C:\\Users\\user\\Zotero\\storage\\7XAHNSYJ\\Fa...,NaN,© 2023 The Authors. Journal of Management Stu...,"artificial intelligence, professional service ..."
3,BIY6Q3U2,journalArticle,2018.0,"Dietvorst, Berkeley J.","Simmons, Joseph P.","Massey, Cade",NaN,NaN,NaN,NaN,...,64.0,NaN,NaN,DOI.org (Crossref),NaN,NaN,C:\\Users\\user\\Zotero\\storage\\KBX873X7\\Di...,NaN,"See discussions, stats, and author profiles fo...","decision making, decision aids, heuristics and..."
4,J83BITLT,journalArticle,2021.0,"Dahlman, Sara","Gulbrandsen, Ib T","Just, Sine N",NaN,NaN,NaN,NaN,...,8.0,NaN,NaN,DOI.org (Crossref),14,NaN,C:\\Users\\user\\Zotero\\storage\\38JAVGYK\\Da...,NaN,Original Research Article\nAlgorithms as organ...,"Algorithms,figuration,fintech,sociotechnical a..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489,G3A58V3U,book,2024.0,"Chen, Y.","Rui, H.","Whinston, A.B.",NaN,NaN,NaN,NaN,...,NaN,Information Systems Research,NaN,NaN,2,NaN,NaN,NaN,NaN,"conversation analytics, predictive analytics,..."
490,XXNGY5UU,journalArticle,2024.0,"Jia, N.","Luo, X.","Fang, Z.","Liao, C.",NaN,NaN,NaN,...,67,NaN,NaN,NaN,56,NaN,NaN,NaN,NaN,NaN
491,67IFMXDZ,journalArticle,2024.0,"Faulconbridge, J.R.","Sarwar, A.","Spring, M.",NaN,NaN,NaN,NaN,...,p.01708406241252930,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,"Accounting, algorithms, artificial intelligenc..."
492,NQCHG2W4,book,2024.0,"Jansen, M.","Nguyen, H.Q.","Shams, A.",NaN,NaN,NaN,NaN,...,NaN,Management Science,NaN,NaN,30,NaN,NaN,NaN,NaN,"lending, underwriting, FinTech, automation, ho..."


In [ ]:
def fetch_affiliation(author_name):
    if not author_name or pd.isna(author_name):
        return None

    search_url = f"https://scholar.google.com/citations?hl=ko&view_op=search_authors&mauthors={author_name.replace(' ', '+')}&btnG="
    response = requests.get(search_url, headers={'User-Agent': 'Mozilla/5.0'})
    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.content, 'html.parser')

    # Convert BeautifulSoup object to lxml object
    dom = etree.HTML(str(soup))

    # Use XPath to find the specific element containing the author's affiliation
    try:
        affiliation_element = dom.xpath('/html/body/div/div[7]/div[2]/div/div/div/div/div[1]')
        if affiliation_element and len(affiliation_element) > 0 and affiliation_element[0].text:
            return affiliation_element[0].text.strip()
    except Exception as e:
        print(f"Error fetching affiliation for {author_name}: {e}")

    return None

# Test the function
author_name = "Massey, Cade	"
affiliation = fetch_affiliation(author_name)
print(f"Author: {author_name}")
print(f"Affiliation: {affiliation}")

Author: Massey, Cade	
Affiliation: University of Pennsylvania


In [ ]:
# Replace None with empty strings for all author columns
for col_name in df.columns:
    if col_name.startswith("Author"):
        df[col_name] = df[col_name].fillna("")

# Apply the function to fetch affiliations for each author column
for col_name in df.columns:
    if col_name.startswith("Author"):
        aff_col_name = f'{col_name} Affiliation'
        df[aff_col_name] = df[col_name].apply(lambda x: fetch_affiliation(x) if x != "" else None)



In [ ]:
df[['Author 1',	'Author 2',	'Author 3',	'Author 4','Author 1 Affiliation','Author 2 Affiliation','Author 3 Affiliation','Author 4 Affiliation' ]]

,Author 1,Author 2,Author 3,Author 4,Author 1 Affiliation,Author 2 Affiliation,Author 3 Affiliation,Author 4 Affiliation
0,"Ferràs-Hernández, Xavier",,,,ESADE Business School - Universitat Ramon Llull,None,None,None
1,"Fayard, Anne-Laure",,,,NYU,None,None,None
2,"Faulconbridge, James","Sarwar, Atif","Spring, Martin",,Lancaster University,"Assistant Professor, Shifa College of Pharmace...","Professor of Operations Management, Lancaster ...",None
3,"Dietvorst, Berkeley J.","Simmons, Joseph P.","Massey, Cade",,None,Institute of High Performance Computing,University of Pennsylvania,None
4,"Dahlman, Sara","Gulbrandsen, Ib T","Just, Sine N",,Roskilde University,"Associate Professor, Roskilde University",Roskilde Uiversity,None
...,...,...,...,...,...,...,...,...
489,"Chen, Y.","Rui, H.","Whinston, A.B.",,Department of Computer Information and Network...,Cornell University,None,None
490,"Jia, N.","Luo, X.","Fang, Z.","Liao, C.","Chief Scientist of Megvii, Managing Director o...","Chair Professor, City University of Hong Kong",Florida State University,None
491,"Faulconbridge, J.R.","Sarwar, A.","Spring, M.",,Lancaster University,NCBA&E Bahawalpur Sub-Campus. Model Town,cold,None
492,"Jansen, M.","Nguyen, H.Q.","Shams, A.",,University of Washington,Kaiser Permanente Southern California,"Department of Psychiatry,",None


In [ ]:
# Create a new order of columns
fixed_columns = ["Key", "Item Type", "Publication Year"]
author_columns = [f"Author {i}" for i in range(1, 15)]
affiliation_columns = [f"Author {i} Affiliation" for i in range(1, 15)]

new_order = fixed_columns
for author, affiliation in zip(author_columns, affiliation_columns):
    if author in df.columns:
        new_order.append(author)
    if affiliation in df.columns:
        new_order.append(affiliation)

# Reorder the DataFrame columns
df = df[new_order]
df

,Key,Item Type,Publication Year,Author 1,Author 1 Affiliation,Author 2,Author 2 Affiliation,Author 3,Author 3 Affiliation,Author 4,...,Author 10,Author 10 Affiliation,Author 11,Author 11 Affiliation,Author 12,Author 12 Affiliation,Author 13,Author 13 Affiliation,Author 14,Author 14 Affiliation
0,54ZC89CL,journalArticle,2018.0,"Ferràs-Hernández, Xavier",ESADE Business School - Universitat Ramon Llull,,None,,None,,...,,None,,None,,None,,None,,None
1,AX85B2EL,journalArticle,2021.0,"Fayard, Anne-Laure",NYU,,None,,None,,...,,None,,None,,None,,None,,None
2,U9TGTFQL,journalArticle,2023.0,"Faulconbridge, James",Lancaster University,"Sarwar, Atif","Assistant Professor, Shifa College of Pharmace...","Spring, Martin","Professor of Operations Management, Lancaster ...",,...,,None,,None,,None,,None,,None
3,BIY6Q3U2,journalArticle,2018.0,"Dietvorst, Berkeley J.",None,"Simmons, Joseph P.",Institute of High Performance Computing,"Massey, Cade",University of Pennsylvania,,...,,None,,None,,None,,None,,None
4,J83BITLT,journalArticle,2021.0,"Dahlman, Sara",Roskilde University,"Gulbrandsen, Ib T","Associate Professor, Roskilde University","Just, Sine N",Roskilde Uiversity,,...,,None,,None,,None,,None,,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489,G3A58V3U,book,2024.0,"Chen, Y.",Department of Computer Information and Network...,"Rui, H.",Cornell University,"Whinston, A.B.",None,,...,,None,,None,,None,,None,,None
490,XXNGY5UU,journalArticle,2024.0,"Jia, N.","Chief Scientist of Megvii, Managing Director o...","Luo, X.","Chair Professor, City University of Hong Kong","Fang, Z.",Florida State University,"Liao, C.",...,,None,,None,,None,,None,,None
491,67IFMXDZ,journalArticle,2024.0,"Faulconbridge, J.R.",Lancaster University,"Sarwar, A.",NCBA&E Bahawalpur Sub-Campus. Model Town,"Spring, M.",cold,,...,,None,,None,,None,,None,,None
492,NQCHG2W4,book,2024.0,"Jansen, M.",University of Washington,"Nguyen, H.Q.",Kaiser Permanente Southern California,"Shams, A.","Department of Psychiatry,",,...,,None,,None,,None,,None,,None
